# KTP Associate Skills Exercise
**Nature Broking Ltd × University of Strathclyde**

---

This exercise assesses your ability to apply natural language processing and textual analysis to corporate sustainability disclosures. It is designed to take approximately **2–3 hours**.

You will:
- Characterise a corpus of US energy sector 10-K filings — Task 1
- Design and implement a Carbon Disclosure Specificity Index — Task 2
- Conduct your own investigation into a question of your choice — Task 3
- Write a short critical synthesis connecting your findings to the voluntary carbon market

**AI assistance is permitted.** After each task you will complete a brief disclosure cell documenting any AI use: what you asked, what it suggested, and what you accepted, modified, or rejected. The panel will use these disclosures as the basis for follow-up questions at interview.

**Before you do anything else, run Step 0.**

In [ ]:
# ── Step 0: Environment Setup ─────────────────────────────────────────────
# Run this cell at the start of every new Colab session.
# It clones the exercise repository and loads all required libraries.
# Colab resets its environment on disconnect — always run this cell first.

import subprocess, os, sys

REPO_URL = "https://github.com/iamjamesbowden/KTP-Exercise.git"
REPO_DIR = "/content/KTP-Exercise"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repository cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("Repository updated.")

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "scikit-learn", "nltk", "wordcloud", "-q"],
    check=True
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import nltk
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

for res in ["punkt", "punkt_tab", "stopwords", "wordnet",
            "averaged_perceptron_tagger"]:
    nltk.download(res, quiet=True)

CORPUS_PATH  = f"{REPO_DIR}/data/corpus.csv"
LM_PATH      = f"{REPO_DIR}/data/lm_dictionary.csv"
HARVARD_PATH = f"{REPO_DIR}/data/harvard_iv_dictionary.csv"

corpus = pd.read_csv(CORPUS_PATH)

print(f"\nCorpus loaded: {len(corpus):,} rows")
print(f"  Firms     : {corpus['firm'].nunique()}")
print(f"  Categories: {sorted(corpus['category'].unique())}")
print(f"  Sections  : {sorted(corpus['section'].unique())}")
print(f"  Years     : {sorted(corpus['year'].unique())}")

APPLICANT_NAME = input("\nEnter your full name: ").strip()
print(f"\nWelcome, {APPLICANT_NAME}. You are ready to begin.")

---

## Research Context

### The Voluntary Carbon Market

The voluntary carbon market (VCM) enables organisations to purchase carbon credits to compensate for greenhouse gas emissions that cannot yet be eliminated through operational decarbonisation. Unlike compliance markets — such as the EU Emissions Trading System — participation in the VCM is driven by corporate sustainability commitments: net zero pledges, Science Based Targets (SBTi), and TCFD-aligned disclosure requirements, rather than by regulatory obligation.

In recent years the quality and credibility of VCM activity has come under significant scrutiny. Investigative reporting and academic research have raised concerns about offset projects that deliver substantially fewer emissions reductions than claimed, and about corporations using carbon credits as a substitute for genuine decarbonisation rather than as a last-resort mechanism for residual emissions. This has prompted the emergence of market integrity bodies — notably the Integrity Council for the Voluntary Carbon Market (ICVCM) and the Voluntary Carbon Markets Integrity Initiative (VCMI) — which publish standards for what constitutes a credible, high-quality carbon credit and a credible corporate claim.

### Nature Broking Ltd

Nature Broking is a carbon credit brokerage firm that curates and brokers premium voluntary carbon credits for corporate clients. Rather than offering transactional, volume-driven credit sales, Nature Broking positions itself as a strategic partner: conducting due diligence aligned to ICVCM standards, constructing diversified portfolios of high-quality credits, and advising clients on how to align their offset strategies with credible, science-based approaches. Their clients are corporations pursuing net zero commitments who need confidence that the credits they purchase represent genuine, permanent, and additional emissions reductions — and that their claims will withstand public and regulatory scrutiny.

### Research Framing

A central challenge for firms operating in or adjacent to the VCM is assessing the seriousness of corporate climate commitments from public disclosure documents. Corporate filings vary enormously in how specifically and verifiably they discuss their carbon strategies — some provide detailed, quantified, and framework-referenced disclosures; others offer only vague aspirational language. Understanding this variation, and what it signals about genuine commitment versus surface-level compliance, is directly relevant to how Nature Broking identifies and advises its clients.

The question motivating this exercise is: **how specifically and credibly do US energy sector firms disclose their carbon and climate strategies in public filings, and what does the language of these disclosures reveal about the seriousness of their commitments?**

---

## About the Corpus

The corpus contains 10-K annual report filings from **30 US energy sector firms** spanning **2019–2023**. Each row represents one section of one filing. Two sections are included per filing where available:

- **`item_1a`** — Risk Factors: the firm's disclosure of material risks, including climate, regulatory, and transition risks
- **`item_7`** — Management Discussion and Analysis (MD&A): narrative discussion of financial performance and operational context

Firms are grouped into three categories:

| Category | Description |
|---|---|
| `oil_gas` | Upstream/midstream oil and gas producers (ExxonMobil, Chevron, ConocoPhillips, Devon Energy, EOG Resources, Hess, Kinder Morgan, Marathon Oil, Occidental Petroleum, Pioneer Natural Resources, Valero Energy, Williams Companies) |
| `utility` | Diversified electric and gas utilities (Ameren, American Electric Power, Dominion Energy, Duke Energy, Entergy, Exelon, NextEra Energy, PPL Corporation, Sempra Energy, Southern Company) |
| `renewable` | Pure-play renewable energy firms (Bloom Energy, Clearway Energy, Enphase Energy, First Solar, Plug Power, Sunnova Energy, SunPower, Sunrun) |

**Key columns:** `firm`, `ticker`, `category`, `year`, `section`, `text`, `word_count`

Two supplementary files are available in `data/` if you wish to use them:
- `lm_dictionary.csv` — Loughran-McDonald financial sentiment dictionary
- `harvard_iv_dictionary.csv` — Harvard General Inquirer dictionary

---

## Task 1: Corpus Characterisation

Before analysing any corpus, it is essential to understand its structure, coverage, and vocabulary. This task asks you to produce three specific outputs. The required outputs are prescribed; how you produce them is up to you.

*Suggested time: 30–45 minutes.*

### Output 1.1 — Filing Inventory

Produce a matrix showing filing coverage across firms and years, grouped by category. Your output should make it immediately clear which firm-year combinations are present in the corpus and whether both sections (Item 1A and Item 7) are available for each.

A reader should be able to identify any gaps in coverage at a glance.

In [ ]:
# Output 1.1 — Filing Inventory
# Your code here


### Output 1.2 — Carbon Vocabulary Frequency

Track the frequency of carbon- and climate-related terminology in the corpus over time, broken down by firm category. The goal is to understand how the language of climate and carbon has evolved across different types of energy firm between 2019 and 2023.

A seed vocabulary is provided below. You may extend it — if you do, note your additions in the AI disclosure cell (or here, if no AI was used). Produce at least one visualisation.

**Seed vocabulary**

*Core:* `carbon offset`, `carbon credit`, `carbon neutral`, `carbon neutrality`, `net zero`, `net-zero`, `Scope 1`, `Scope 2`, `Scope 3`, `TCFD`, `SBTi`, `science-based target`, `nature-based solutions`, `sequestration`, `carbon capture`, `greenhouse gas`, `GHG`, `emissions reduction`, `climate target`, `climate commitment`, `carbon footprint`

*VCM-specific:* `Article 6`, `CORSIA`, `REDD+`, `Gold Standard`, `Verra`, `ICVCM`, `VCMI`, `insetting`, `additionality`, `permanence`, `leakage`, `biodiversity credit`, `carbon removal`, `CDR`, `verified carbon`

> **Note:** Think carefully about how you count term frequency. Raw counts will favour longer documents. Consider whether normalisation is appropriate and, if so, how.

In [ ]:
# Output 1.2 — Carbon Vocabulary Frequency
# Your code here


### Output 1.3 — Disclosure Volume Trends

Examine how the volume of climate-relevant text has changed over time across firm categories. Are firms writing more or less about climate and carbon? Does this differ between oil & gas, utilities, and renewables?

Produce at least one visualisation. Consider briefly — in a comment or a markdown cell — what the trends suggest about how disclosure behaviour has evolved over the period, and whether volume alone is a meaningful indicator of the seriousness of a firm's climate engagement.

In [ ]:
# Output 1.3 — Disclosure Volume Trends
# Your code here


---

## AI Assistance Disclosure — Task 1

**Did you use an AI assistant for any part of Task 1?** *(double-click this cell to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete the table below for each instance. Leave unused rows blank.*

| # | Tool used | What you asked it to do | What it suggested | Accepted / modified / rejected | Your reasoning |
|---|-----------|------------------------|-------------------|-------------------------------|----------------|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach to this task? What decisions did you make independently?)*

> *Your reflection here.*

---

## Task 2: Carbon Disclosure Specificity Index

*Suggested time: 45–60 minutes.*

### Background

In the voluntary carbon market, there is a recognised and consequential distinction between two types of corporate climate language:

**Vague / aspirational**
> *"We are committed to reducing our environmental impact and achieving carbon neutrality."*

**Specific / verifiable**
> *"In 2022 we retired 1.8 million tonnes CO₂e of Gold Standard-verified nature-based carbon credits, representing 23% of our Scope 3 emissions. Our offset portfolio is aligned with ICVCM Core Carbon Principles and is independently audited annually."*

The first statement conveys intent without accountability. The second provides quantification, a verification standard, a credit type, and scope coverage — all of which are necessary for a credible VCM claim. The gap between these two types of disclosure is a central preoccupation of market integrity bodies such as ICVCM and VCMI, and is directly relevant to how Nature Broking advises its clients.

### Your Task

Design and implement a **Carbon Disclosure Specificity Index (CDSI)**: a continuous score, computed per firm-year, measuring how specifically and credibly each filing discusses the firm's carbon and climate strategy.

There is no single correct approach. You might consider:
- A lexicon-based approach using terms that signal specificity (quantification, named verification standards, explicit frameworks) versus vagueness (generic commitments, unanchored modal verbs)
- A rule-based system based on structural or syntactic patterns
- A sentence embedding or similarity-based approach
- An LLM-assisted scoring approach
- A hybrid of two or more of the above

Whatever approach you choose, it must be **implemented in code** and must produce a **numeric score for each firm-year observation**.

### Required Outputs

1. A pivot table of CDSI scores with firms as rows and years as columns, grouped by category
2. At least one visualisation showing how scores vary across categories and/or over time
3. A written methodological justification (200–300 words) in the cell provided below

In [ ]:
# Task 2 — CDSI implementation
# Your code here


In [ ]:
# Task 2 — Pivot table and visualisation
# Your code here


### Methodological Justification

*(Double-click to edit — 150–200 words.)*

In 150–200 words, explain: the approach you took and why, the key assumptions it makes, and its main limitations.

> *Your justification here.*

---

## AI Assistance Disclosure — Task 2

**Did you use an AI assistant for any part of Task 2?** *(double-click to edit)*

- [ ] Yes
- [ ] No

*If yes, complete the table below for each instance. Leave unused rows blank.*

| # | Tool used | What you asked it to do | What it suggested | Accepted / modified / rejected | Your reasoning |
|---|-----------|------------------------|-------------------|-------------------------------|----------------|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

**Overall reflection** *(2–3 sentences)*

> *Your reflection here.*

---

## Task 3: Open Investigation

*Suggested time: 45–60 minutes.*

### Instructions

Design and conduct your own investigation using the Scenario A corpus. This task is an opportunity to demonstrate the analytical questions you find interesting and the methods you are most confident applying.

**Guardrails:**
- You must use the `corpus` dataframe loaded in Step 0
- Your research question must connect to at least one aspect of corporate voluntary carbon market behaviour, climate disclosure quality, or sustainability strategy
- You must produce at least one quantitative output — a table, chart, or computed statistic
- You must provide a written interpretation of your findings (300–400 words)

Strong responses will show original thinking, clear methodological reasoning, and an ability to situate findings within the VCM context introduced in the briefing. You are not expected to produce a complete research paper — focus on demonstrating how you approach and think through an analytical problem.

### Research Question and Justification

*(Double-click to edit.)*

**My research question:**
> *State your question here.*

**Why this question is relevant to Nature Broking's business context** *(2–3 sentences):*
> *Your justification here.*

In [ ]:
# Task 3 — Your code here


In [ ]:
# Task 3 — continued


In [ ]:
# Task 3 — continued


### Interpretation of Findings

*(Double-click to edit — 100–150 words.)*

In 100–150 words, summarise what your results show, what they suggest about corporate VCM behaviour in this corpus, and one logical next step.

> *Your interpretation here.*

---

## AI Assistance Disclosure — Task 3

**Did you use an AI assistant for any part of Task 3?** *(double-click to edit)*

- [ ] Yes
- [ ] No

*If yes, complete the table below for each instance. Leave unused rows blank.*

| # | Tool used | What you asked it to do | What it suggested | Accepted / modified / rejected | Your reasoning |
|---|-----------|------------------------|-------------------|-------------------------------|----------------|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

**Overall reflection** *(2–3 sentences)*

> *Your reflection here.*

---

## Written Synthesis

*(Double-click to edit — 150–200 words.)*

Draw together the key patterns from Tasks 1, 2, and 3. Address:
- What the analysis reveals collectively about climate disclosure in the US energy sector over 2019–2023
- The main limitations of your approach across the three tasks
- How you would improve the analysis given more time or data

> *Your synthesis here.*